# Central Limit Theorem

_Probability & Statistics_

**Add up many independent things and the total looks Normal — no matter where they came from.**

Here's the magic: average many independent random things, and the average looks like a bell curve.

     This happens even if each thing came from a weird, non-bell-shaped distribution.

---

This notebook builds the **Central Limit Theorem** demonstration one piece at a time: first we average many batches of uniform draws, then we test how Normal the result is, then we picture both the bell shape and the shrinking spread. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPy

We start from the decidedly non-bell-shaped **Uniform(0, 1)** distribution and average it in batches of 30. We build the demo in three steps: (1) generate the sample means, (2) compare their mean and spread to what the CLT predicts, (3) formally test that they look Normal.

### Step 1 — Average many batches of uniform draws

The CLT is about the distribution of an **average**. So we build a big table where each of the 200,000 rows holds 30 independent Uniform(0, 1) draws, then collapse each row to its mean.

A single Uniform(0, 1) draw has mean 0.5 and variance 1/12. We keep those numbers handy because the CLT will predict the spread of the averages from them.

In [ ]:
rng = np.random.default_rng(0)
n = 30                              # samples per average
trials = 200000                    # how many averages we form

# Each row holds n Uniform(0,1) draws; collapse each row to its mean.
data = rng.uniform(0, 1, size=(trials, n))
means = data.mean(axis=1)
print("formed", trials, "sample means, each averaging", n, "draws")

### Step 2 — Check the predicted center and spread

The CLT predicts the averages are centered at the original mean (0.5) with a standard deviation that shrinks as $\sqrt{\sigma^2 / n}$ — here $\sqrt{(1/12)/30}$. We compare the empirical mean and standard deviation of our 200,000 averages against those predictions; they should line up closely.

In [ ]:
# CLT predicts means ~ Normal(0.5, sqrt((1/12)/n)).
mu = 0.5
sd = np.sqrt((1 / 12) / n)

print("sample mean =", round(means.mean(), 4), " predicted", mu)
print("sample std  =", round(means.std(), 4), " predicted", round(sd, 4))

### Step 3 — Formally test for Normality

Matching the mean and spread is necessary but not enough — we want the whole *shape* to be Normal. We standardize the averages (subtract `mu`, divide by `sd`) so they should look like a standard Normal, then run a **Kolmogorov–Smirnov** test against `norm.cdf`. A large p-value means we cannot tell the averages apart from a true bell curve — exactly what the CLT promises.

In [ ]:
from scipy.stats import kstest, norm

# Standardized means should match a standard normal (big p = good fit).
z = (means - mu) / sd
stat, pval = kstest(z, norm.cdf)
print("KS stat =", round(stat, 4), " p-value =", round(pval, 4))

## Visualize the data & results

_How does the central limit theorem shape the distribution of sample means?_

### Step 1 — Regenerate the sample means

We rebuild the same 200,000 averages of 30 Uniform(0, 1) draws so this plotting cell stands on its own. These averages are what the left-hand histogram will show.

In [ ]:
# Sampling distribution of the mean of 30 Uniform(0,1) draws.
rng = np.random.default_rng(0)
data = rng.uniform(0, 1, size=(200000, 30))
means = data.mean(axis=1)

### Step 2 — Compute how the spread shrinks with n

The CLT also says the spread of the average shrinks as $1/\sqrt{n}$. For a range of batch sizes `n`, we compute the predicted standard deviation $\sqrt{(1/12)/n}$ — the right-hand plot will trace this decay curve.

In [ ]:
# Std of the sample mean shrinks as 1/sqrt(n).
ns = np.array([1, 2, 5, 10, 20, 30, 50, 100])
sds = np.sqrt((1 / 12) / ns)

### Step 3 — Plot the bell shape and the shrinking spread

Left: the histogram of the averages — even though each ingredient was flat Uniform noise, the averages pile up into a bell curve. Right: the standard deviation of the average plotted against `n`, showing the $1/\sqrt{n}$ shrinkage that makes larger samples more precise.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left — histogram of the sample means: a bell curve emerges.
ax1.hist(means, bins=40, density=True, color='#7ee787')
ax1.set_title('Sampling distribution of the mean of 30 Uniform(0,1) draws')

# Right — spread of the sample mean shrinks as 1/sqrt(n).
ax2.plot(ns, sds, color='#4ea1ff', marker='o')
ax2.set_title('Spread of the sample mean shrinks as 1/sqrt(n)')
ax2.set_xlabel('samples per average n')
ax2.set_ylabel('std of sample mean')

plt.show()